In [24]:
import pandas as pd
import numpy as np

# 1. Load the Core Files
results = pd.read_csv('data/MRegularSeasonDetailedResults.csv')
teams = pd.read_csv('data/MTeams.csv')
seeds = pd.read_csv('data/MNCAATourneySeeds.csv')

# 2. Clean Seeds: Convert 'W01' -> 1 (Integer)
# This allows the model to calculate "Seed Difference" mathematically.
seeds['Seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))

# 3. Create Season-Long Averages for every team
winning_stats = results.groupby(['Season', 'WTeamID'])[['WScore', 'WFGM', 'WAst']].mean().rename(
    columns={'WScore': 'Score', 'WFGM': 'FGM', 'WAst': 'Ast'})
losing_stats = results.groupby(['Season', 'LTeamID'])[['LScore', 'LFGM', 'LAst']].mean().rename(
    columns={'LScore': 'Score', 'LFGM': 'FGM', 'WAst': 'Ast'})

# Combine and group by Season/Team to get one row per team per year
team_stats = pd.concat([winning_stats, losing_stats]).groupby(level=[0, 1]).mean()

# 4. Filter for the most recent season stats to use for the 2026 Bracket
# This ensures we are predicting based on current team performance.
latest_season = team_stats.index.get_level_values(0).max()
latest_stats = team_stats.xs(latest_season, level=0)

print(f"✅ Step 1 Complete: Data prepared for Season {latest_season}.")

✅ Step 1 Complete: Data prepared for Season 2026.


In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Merge Seeds into your Season Results
# We need to see what the seeds were for every game played in tournament history to train
train_results = results.merge(seeds, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'])
train_results = train_results.rename(columns={'Seed': 'WSeed'}).drop('TeamID', axis=1)
train_results = train_results.merge(seeds, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'])
train_results = train_results.rename(columns={'Seed': 'LSeed'}).drop('TeamID', axis=1)

# 2. Build the Features (Difference in Stats + Difference in Seeds)
side_a = pd.DataFrame()
side_a['ScoreDiff'] = train_results['WScore'] - train_results['LScore']
side_a['FGM_Diff'] = train_results['WFGM'] - train_results['LFGM']
side_a['Ast_Diff'] = train_results['WAst'] - train_results['LAst']
side_a['SeedDiff'] = train_results['WSeed'] - train_results['LSeed']
side_a['Result'] = 1

side_b = pd.DataFrame()
side_b['ScoreDiff'] = train_results['LScore'] - train_results['WScore']
side_b['FGM_Diff'] = train_results['LFGM'] - train_results['WFGM']
side_b['Ast_Diff'] = train_results['LAst'] - train_results['WAst']
side_b['SeedDiff'] = train_results['LSeed'] - train_results['WSeed']
side_b['Result'] = 0

train_df = pd.concat([side_a, side_b]).dropna()

# 3. Scaling & Final Training
scaler = StandardScaler()
features = ['ScoreDiff', 'FGM_Diff', 'Ast_Diff', 'SeedDiff']
X = scaler.fit_transform(train_df[features])
y = train_df['Result']

# Using C=0.01 keeps the model from being "overconfident"
model = LogisticRegression(C=0.01)
model.fit(X, y)

print("✅ Step 2 Complete: Your model now respects the power of seeds!")

✅ Step 2 Complete: Your model now respects the power of seeds!


In [34]:
# 1. THE FINAL NAME TRANSLATOR
name_map = {
    "St. John's": "St John's",
    "Miami (Florida)": "Miami FL",
    "UConn": "Connecticut",
    "Ohio State": "Ohio St",
    "Michigan State": "Michigan St",
    "North Dakota State": "N Dakota St",
    "LIU": "LIU Brooklyn",
    "Utah State": "Utah St",
    "Kennesaw State": "Kennesaw",
    "Prairie View A&M": "PV A&M",
    "Wright State": "Wright St",
    "Iowa State": "Iowa St",
    "Tennessee State": "Tennessee St",
    "McNeese": "McNeese St",
    "Gonzaga": "Gonzaga",
    "Florida": "Florida"
}

# 2. THE PREDICTION ENGINE
def predict_game_v4(team1_name, team2_name, s1_seed, s2_seed):
    t1_lookup = name_map.get(team1_name, team1_name)
    t2_lookup = name_map.get(team2_name, team2_name)

    try:
        t1_id = teams[teams['TeamName'] == t1_lookup]['TeamID'].values[0]
        t2_id = teams[teams['TeamName'] == t2_lookup]['TeamID'].values[0]
        # Use average of available stats
        s1 = team_stats.loc[(slice(None), t1_id), :].mean()
        s2 = team_stats.loc[(slice(None), t2_id), :].mean()
    except:
        return None

    # Strength of Schedule Correction (2.0 makes the model respect the Power 5)
    seed_weight = 2.0

    diff = pd.DataFrame([[
        s1['Score'] - s2['Score'],
        s1['FGM'] - s2['FGM'],
        s1['Ast'] - s2['Ast'],
        (s1_seed - s2_seed) * seed_weight
    ]], columns=['ScoreDiff', 'FGM_Diff', 'Ast_Diff', 'SeedDiff'])

    diff_scaled = scaler.transform(diff)
    return model.predict_proba(diff_scaled)[0][1]

# 3. RUNNING THE BRACKET
print(f"{'2026 OFFICIAL FIRST ROUND MATCHUP':<50} | {'WINNER PROBABILITY'}")
print("-" * 85)

for t1, t2, s1, s2 in bracket_matchups:
    prob = predict_game_v4(t1, t2, s1, s2)

    if prob is not None:
        if prob >= 0.5:
            winner, win_prob = t1, prob
        else:
            winner, win_prob = t2, (1 - prob)

        print(f"{t1 + ' (' + str(s1) + ') vs ' + t2 + ' (' + str(s2) + ')':<50} | {winner} ({win_prob:.2%})")
    else:
        print(f"FAILED: Check mapping for {t1} or {t2}")

print("-" * 85)

2026 OFFICIAL FIRST ROUND MATCHUP                  | WINNER PROBABILITY
-------------------------------------------------------------------------------------
Duke (1) vs Siena (16)                             | Duke (98.48%)
Ohio State (8) vs TCU (9)                          | Ohio State (60.61%)
St. John's (5) vs Northern Iowa (12)               | St. John's (91.67%)
Kansas (4) vs Cal Baptist (13)                     | Kansas (87.76%)
Louisville (6) vs South Florida (11)               | Louisville (89.48%)
Michigan State (3) vs North Dakota State (14)      | Michigan State (77.03%)
UCLA (7) vs UCF (10)                               | UCLA (82.73%)
UConn (2) vs Furman (15)                           | UConn (91.16%)
Arizona (1) vs LIU (16)                            | Arizona (94.61%)
Villanova (8) vs Utah State (9)                    | Villanova (62.65%)
Wisconsin (5) vs High Point (12)                   | High Point (58.48%)
Arkansas (4) vs Hawaii (13)                        | Arkansa